# HW 10

## Imports

In [59]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, cross_val_score

## 1. The UC Irvine machine learning data repository hosts a collection of data on whether a mushroom is edible, donated by Jeff Schlimmer and to be found at http://archive.ics.uci.edu/ml/datasets/Mushroom. This data has a set of categorical attributes of the mushroom, together with two labels (poisonous or edible). Use the R random forest package (as in the example in the chapter) to build a random forest to classify a mushroom as edible or poisonous based on its attributes.

## Produce a class-confusion matrix for this problem. If you eat a mushroom based on your classifier’s prediction it is edible, what is the probability of being poisoned?

In [2]:
# fetch dataset 
mushroom = fetch_ucirepo(id=73)  
# data (as pandas dataframes) 
mushrooms = mushroom.data.original
mushrooms

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,e
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,k,s,n,f,n,a,c,b,y,e,...,o,o,p,o,o,p,b,c,l,e
8120,x,s,n,f,n,a,c,b,y,e,...,o,o,p,n,o,p,b,v,l,e
8121,f,s,n,f,n,a,c,b,n,e,...,o,o,p,o,o,p,b,c,l,e
8122,k,y,n,f,y,f,c,n,b,t,...,w,w,p,w,o,e,w,v,l,p


In [3]:
mushrooms['poisonous'] = mushrooms['poisonous'].apply(lambda x: x in ['p'])
mushrooms['poisonous'] = mushrooms['poisonous'].astype(int)

In [4]:
mushrooms

,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,1
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,0
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,0
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,1
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,k,s,n,f,n,a,c,b,y,e,...,o,o,p,o,o,p,b,c,l,0
8120,x,s,n,f,n,a,c,b,y,e,...,o,o,p,n,o,p,b,v,l,0
8121,f,s,n,f,n,a,c,b,n,e,...,o,o,p,o,o,p,b,c,l,0
8122,k,y,n,f,y,f,c,n,b,t,...,w,w,p,w,o,e,w,v,l,1


In [16]:
X = mushrooms.drop('poisonous', axis =1)
X = pd.get_dummies(X)
y = mushrooms['poisonous']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
X_train

,cap-shape_b,cap-shape_c,cap-shape_f,cap-shape_k,cap-shape_s,cap-shape_x,cap-surface_f,cap-surface_g,cap-surface_s,cap-surface_y,...,population_s,population_v,population_y,habitat_d,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
7873,False,False,False,True,False,False,False,False,True,False,...,False,True,False,True,False,False,False,False,False,False
6515,False,False,False,False,False,True,False,False,True,False,...,False,True,False,False,False,False,False,True,False,False
6141,False,False,True,False,False,False,False,False,False,True,...,False,True,False,False,False,True,False,False,False,False
2764,False,False,True,False,False,False,True,False,False,False,...,False,True,False,True,False,False,False,False,False,False
438,True,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5226,False,False,False,False,False,True,False,False,False,True,...,False,True,False,False,False,False,False,True,False,False
5390,False,False,False,True,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
860,False,False,True,False,False,False,False,False,False,True,...,False,False,True,False,False,False,False,True,False,False
7603,False,False,False,True,False,False,False,False,True,False,...,False,True,False,False,False,False,False,True,False,False


In [29]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
rf_train = rf.predict(X_train)
rf_test = rf.predict(X_test)

In [30]:
accuracy_test = accuracy_score(y_test, rf_test)
print(accuracy_test)

0.92456608811749


In [31]:
tn, fp, fn, tp = confusion_matrix(y_test, rf_test).ravel()
print(fn)

160


According to my model, The probability that I eat an edible mushroom and it turns out to be poisononous is 0%.

##  2. Build a decision tree with a depth of 50 for the following dataset. In addition, build a random forest classifier with 100 estimators and a depth of 50 for the following dataset EEG Eye State

In [56]:
df = pd.read_csv("EEG_Eye_State_formatted-1.csv")
df

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4,eyeDetection
0,4329.23,4009.23,4289.23,4148.21,4350.26,4586.15,4096.92,4641.03,4222.05,4238.46,4211.28,4280.51,4635.90,4393.85,0
1,4324.62,4004.62,4293.85,4148.72,4342.05,4586.67,4097.44,4638.97,4210.77,4226.67,4207.69,4279.49,4632.82,4384.10,0
2,4327.69,4006.67,4295.38,4156.41,4336.92,4583.59,4096.92,4630.26,4207.69,4222.05,4206.67,4282.05,4628.72,4389.23,0
3,4328.72,4011.79,4296.41,4155.90,4343.59,4582.56,4097.44,4630.77,4217.44,4235.38,4210.77,4287.69,4632.31,4396.41,0
4,4326.15,4011.79,4292.31,4151.28,4347.69,4586.67,4095.90,4627.69,4210.77,4244.10,4212.82,4288.21,4632.82,4398.46,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14975,4281.03,3990.26,4245.64,4116.92,4333.85,4614.36,4074.87,4625.64,4203.08,4221.54,4171.28,4269.23,4593.33,4340.51,1
14976,4276.92,3991.79,4245.13,4110.77,4332.82,4615.38,4073.33,4621.54,4194.36,4217.44,4162.56,4259.49,4590.26,4333.33,1
14977,4277.44,3990.77,4246.67,4113.85,4333.33,4615.38,4072.82,4623.59,4193.33,4212.82,4160.51,4257.95,4591.79,4339.49,1
14978,4284.62,3991.79,4251.28,4122.05,4334.36,4616.41,4080.51,4628.72,4200.00,4220.00,4165.64,4267.18,4596.41,4350.77,1


In [57]:
X = df.drop('eyeDetection', axis =1)
y = df['eyeDetection']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dfc = DecisionTreeClassifier(max_depth = 50)
dfc.fit(X_train, y_train)
dfc_train = dfc.predict(X_train)
dfc_test = dfc.predict(X_test)
dfc_accuracy = accuracy_score(y_test, dfc_test)

rf = RandomForestClassifier(n_estimators = 100,max_depth = 50)
rf.fit(X_train, y_train)
rf_train = rf.predict(X_train)
rf_test = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_test)

print("Descision Tree:", dfc_accuracy,'\n',"Random Forest:", rf_accuracy)

tn, fp, fn, tp = confusion_matrix(y_test, dfc_test).ravel()
print("Descision Tree [tn, fp, fn, tp]:",tn, fp, fn, tp)

tn, fp, fn, tp = confusion_matrix(y_test, rf_test).ravel()
print("Random Forest [tn, fp, fn, tp]:",tn, fp, fn, tp)

Descision Tree: 0.8397863818424566 
 Random Forest: 0.9195594125500668
Descision Tree [tn, fp, fn, tp]: 1356 230 250 1160
Random Forest [tn, fp, fn, tp]: 1513 73 168 1242


## Why does random forest perform better than the decision tree?

It performs better because Random Forrests combines multiple trees to get its ouput,reducing the risk of overfitting that can happen with just 1 tree.

## Can we change the depth to the extent that decision tree performs much better than random forest? Explain your reasoning with solid evidence.

While increasing the depth might lead to a higher accuracy rate, it would not be as reliable as it can overfit the data to the point where each point can be classified as it's own leaf in the tree. If we then tested the model with new points, it can run the risk of mis classification. As we are dealing with multiple electronodes, it's best to use random forest in this case rather than a single tree.

## 3. Build a decision tree with a depth of 50 for the following dataset. In addition, build a random forest classifier with 100 estimators and a depth of 50 for the following dataset Haberman's Survival

In [46]:
df1 = pd.read_csv("haberman.data")
df1

,Age,Year,positive_axillary_nodes,survival_status
0,30,64,1,1
1,30,62,3,1
2,30,65,0,1
3,31,59,2,1
4,31,65,4,1
...,...,...,...,...
301,75,62,1,1
302,76,67,0,1
303,77,65,3,1
304,78,65,1,2


In [54]:
X1 = df1.drop(' survival_status', axis = 1)
y1 = df1[' survival_status']
X_train, X_test, y_train, y_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

dfc = DecisionTreeClassifier(max_depth = 50)
dfc.fit(X_train, y_train)
dfc_train = dfc.predict(X_train)
dfc_test = dfc.predict(X_test)
dfc_accuracy = accuracy_score(y_test, dfc_test)

rf = RandomForestClassifier(n_estimators = 100,max_depth = 50)
rf.fit(X_train, y_train)
rf_train = rf.predict(X_train)
rf_test = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_test)

print("Descision Tree:", dfc_accuracy,'\n',"Random Forest:", rf_accuracy)

tn, fp, fn, tp = confusion_matrix(y_test, dfc_test).ravel()
print("Descision Tree [tn, fp, fn, tp]:",tn, fp, fn, tp)

tn, fp, fn, tp = confusion_matrix(y_test, rf_test).ravel()
print("Random Forest [tn, fp, fn, tp]:",tn, fp, fn, tp)

Descision Tree: 0.6612903225806451 
 Random Forest: 0.6451612903225806
Descision Tree [tn, fp, fn, tp]: 35 9 12 6
Random Forest [tn, fp, fn, tp]: 37 7 15 3


## a. Why does the dataset work poorly with decision trees and random forests?
The dataset work poorly because this is not the best model to use for classifying based on survival status. It is better to use  a binary classification model as we are choosing between 2 statuses. Additionally, the operation year column might be potentially adding noise to the model.

## b. Can we clearly say if one of these is better than the other? Why or why not?
Based on the testing metrics we can't be certain if one is better than the other as the noisy and inaccurate data impacted the accuracy of both models.

In [47]:
df1.corr()

,Age,Year,positive_axillary_nodes,survival_status
Age,1.000000,0.089529,-0.063176,0.067950
Year,0.089529,1.000000,-0.003764,-0.004768
positive_axillary_nodes,-0.063176,-0.003764,1.000000,0.286768
survival_status,0.067950,-0.004768,0.286768,1.000000


##  4. Build two other classifiers with the same training/evaluation set split as in Problem 2 for that same data set, then do cross-validation with fold=5, finally demonstrate the best model's confusion matrix and accuracy

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dfc = DecisionTreeClassifier(max_depth = 50)
dfc.fit(X_train, y_train)
dfc_train = dfc.predict(X_train)
dfc_test = dfc.predict(X_test)
dfc_accuracy = accuracy_score(y_test, dfc_test)

hgbc = HistGradientBoostingClassifier(max_depth = 50)
hgbc.fit(X_train, y_train)
hgbc_train = hgbc.predict(X_train)
hgbc_test = hgbc.predict(X_test)
hgbc_accuracy = accuracy_score(y_test, hgbc_test)

print("Descision Tree Accuracy:", dfc_accuracy,'\n',"Histogram Gradient Boosting Classifier Accuracy:", hgbc_accuracy)

k_folds = KFold(n_splits = 5)
dfc_scores = cross_val_score(dfc, X, y, cv = k_folds)
hgbc_scores = cross_val_score(hgbc, X, y, cv = k_folds)
print("Histogram Gradient Boosting Classifier CV Scores:",hgbc_scores)

#tn, fp, fn, tp = confusion_matrix(y_test, dfc_test).ravel()
#print("Descision Tree [tn, fp, fn, tp]:",tn, fp, fn, tp)

tn, fp, fn, tp = confusion_matrix(y_test, hgbc_test).ravel()
print("Histogram Gradient Boosting Classifier [tn, fp, fn, tp]:",tn, fp, fn, tp)

Descision Tree Accuracy: 0.8381174899866488 
 Histogram Gradient Boosting Classifier Accuracy: 0.9065420560747663
Histogram Gradient Boosting Classifier CV Scores: [0.60747664 0.50867824 0.22162884 0.37016021 0.41989319]
Histogram Gradient Boosting Classifier [tn, fp, fn, tp]: 1491 95 185 1225
